# InsightFace Benchmark Workflow

This notebook is a local wrapper for the InsightFace/SCRFD benchmark commands. The CLI tools in `src/face_detection_benchmark/` are the source of truth; this notebook only configures paths, checks artifacts, and calls those commands when explicit run flags are enabled.

## Dataset Policy

The cleaned target-domain Roboflow dataset can be used here as validation analysis for choosing an InsightFace operating threshold. If you choose a threshold on this dataset, do not also report the same run as an unbiased test-set benchmark.

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "face_detection_benchmark").exists():
            return candidate
    raise RuntimeError("Could not find repo root from notebook location.")


def repo_path(path: Path) -> Path:
    return path if path.is_absolute() else REPO_ROOT / path


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from face_detection_benchmark import config as benchmark_config
from face_detection_benchmark.models.insightface import (
    DEFAULT_INSIGHTFACE_CTX_ID,
    DEFAULT_INSIGHTFACE_DET_SIZE,
    DEFAULT_INSIGHTFACE_MODEL_NAME,
    DEFAULT_INSIGHTFACE_MODEL_PACK,
    DEFAULT_INSIGHTFACE_PROVIDERS,
    DEFAULT_INSIGHTFACE_THRESHOLD,
)

print(f"Repo root: {REPO_ROOT}")

In [ ]:
insightface_available = importlib.util.find_spec("insightface") is not None
onnxruntime_available = importlib.util.find_spec("onnxruntime") is not None

print(f"InsightFace installed: {insightface_available}")
print(f"ONNX Runtime installed: {onnxruntime_available}")
if not insightface_available or not onnxruntime_available:
    print("Install optional dependencies with: uv sync --extra insightface")

In [ ]:
benchmark_dataset_dir = (
    repo_path(benchmark_config.DEFAULT_BENCHMARK_DATA_DIR)
    / benchmark_config.DEFAULT_BENCHMARK_DATASET_NAME
    / benchmark_config.DEFAULT_ROBOFLOW_TEST_SPLIT
)
annotation_path = benchmark_dataset_dir / "_annotations.coco.json"

print(f"Benchmark dataset dir exists: {benchmark_dataset_dir.exists()} - {benchmark_dataset_dir}")
print(f"COCO annotation file exists: {annotation_path.exists()} - {annotation_path}")
if annotation_path.exists():
    coco_data = json.loads(annotation_path.read_text())
    print(f"COCO images: {len(coco_data.get('images', []))}")
    print(f"COCO annotations: {len(coco_data.get('annotations', []))}")
    print(f"COCO categories: {coco_data.get('categories', [])}")

## Run InsightFace Prediction

This command runs InsightFace/SCRFD on the cleaned COCO split and writes normalized prediction JSONL. The low default threshold keeps candidate detections available for later validation sweeps, but it can create a large prediction file.

In [ ]:
RUN_PREDICTION = False

INSIGHTFACE_RUN_ID = "insightface-buffalo-l-validation"
INSIGHTFACE_MODEL_NAME = DEFAULT_INSIGHTFACE_MODEL_NAME
INSIGHTFACE_MODEL_PACK = DEFAULT_INSIGHTFACE_MODEL_PACK
INSIGHTFACE_THRESHOLD = DEFAULT_INSIGHTFACE_THRESHOLD
INSIGHTFACE_DET_SIZE = DEFAULT_INSIGHTFACE_DET_SIZE
INSIGHTFACE_PROVIDERS = ",".join(DEFAULT_INSIGHTFACE_PROVIDERS)
INSIGHTFACE_CTX_ID = DEFAULT_INSIGHTFACE_CTX_ID
INSIGHTFACE_BATCH_SIZE = 4

INSIGHTFACE_PREDICTIONS_PATH = (
    repo_path(benchmark_config.DEFAULT_RUNS_DIR)
    / "benchmarks"
    / INSIGHTFACE_RUN_ID
    / "predictions"
    / f"{INSIGHTFACE_MODEL_NAME}.jsonl"
)

print(f"Prediction output: {INSIGHTFACE_PREDICTIONS_PATH}")

In [ ]:
if RUN_PREDICTION:
    subprocess.run(
        [
            "uv",
            "run",
            "face-benchmark",
            "predict-insightface-benchmark",
            "--dataset-dir",
            str(benchmark_dataset_dir),
            "--run-id",
            INSIGHTFACE_RUN_ID,
            "--model-name",
            INSIGHTFACE_MODEL_NAME,
            "--model-pack",
            INSIGHTFACE_MODEL_PACK,
            "--threshold",
            str(INSIGHTFACE_THRESHOLD),
            "--det-size",
            str(INSIGHTFACE_DET_SIZE),
            "--providers",
            INSIGHTFACE_PROVIDERS,
            "--ctx-id",
            str(INSIGHTFACE_CTX_ID),
            "--batch-size",
            str(INSIGHTFACE_BATCH_SIZE),
        ],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Skipping InsightFace prediction. Set RUN_PREDICTION = True to run it.")

In [ ]:
prediction_rows = read_jsonl(INSIGHTFACE_PREDICTIONS_PATH)
prediction_box_count = sum(len(row.get("detections", [])) for row in prediction_rows)

print(f"Prediction file exists: {INSIGHTFACE_PREDICTIONS_PATH.exists()} - {INSIGHTFACE_PREDICTIONS_PATH}")
print(f"Prediction rows: {len(prediction_rows)}")
print(f"Prediction boxes: {prediction_box_count}")
if INSIGHTFACE_PREDICTIONS_PATH.exists():
    print(f"Prediction file size MB: {INSIGHTFACE_PREDICTIONS_PATH.stat().st_size / 1_000_000:.1f}")
if prediction_rows:
    prediction_rows[0]

## Run Threshold Validation

Use this step only when you intentionally treat the cleaned split as validation data. This writes threshold tables and SVG plots under `runs/validation/<run-id>/`.

In [ ]:
RUN_VALIDATION = False

VALIDATION_RUN_ID = INSIGHTFACE_RUN_ID
SELECTION_METRIC = "f2"
VALIDATION_DIR = repo_path(benchmark_config.DEFAULT_RUNS_DIR) / "validation" / VALIDATION_RUN_ID

print(f"Validation output dir: {VALIDATION_DIR}")

In [ ]:
if RUN_VALIDATION:
    subprocess.run(
        [
            "uv",
            "run",
            "face-benchmark",
            "validate-thresholds",
            "--dataset-dir",
            str(benchmark_dataset_dir),
            "--predictions-path",
            str(INSIGHTFACE_PREDICTIONS_PATH),
            "--run-id",
            VALIDATION_RUN_ID,
            "--selection-metric",
            SELECTION_METRIC,
        ],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Skipping threshold validation. Set RUN_VALIDATION = True to run it.")

In [ ]:
selected_threshold_path = VALIDATION_DIR / "selected_threshold.json"
threshold_metrics_path = VALIDATION_DIR / "threshold_metrics.md"
precision_recall_path = VALIDATION_DIR / "precision_recall.svg"
f_scores_path = VALIDATION_DIR / "f1_f2_by_threshold.svg"

print(f"Selected threshold exists: {selected_threshold_path.exists()} - {selected_threshold_path}")
print(f"Threshold metrics exists: {threshold_metrics_path.exists()} - {threshold_metrics_path}")
print(f"Precision-recall plot exists: {precision_recall_path.exists()} - {precision_recall_path}")
print(f"F-score plot exists: {f_scores_path.exists()} - {f_scores_path}")
if selected_threshold_path.exists():
    json.loads(selected_threshold_path.read_text())

## Compare Validation Runs

Use this step to compare InsightFace against existing RF-DETR validation runs. The comparison command reads existing `threshold_validation.json` files and writes ignored local comparison plots.

In [ ]:
RUN_COMPARISON = False

COMPARISON_RUN_ID = "model-family-comparison"
VALIDATION_RUNS = [
    repo_path(benchmark_config.DEFAULT_RUNS_DIR) / "validation" / "rfdetr-ema-1-validation",
    repo_path(benchmark_config.DEFAULT_RUNS_DIR) / "validation" / "rfdetr-ema-2-validation",
    VALIDATION_DIR,
]
COMPARISON_DIR = repo_path(benchmark_config.DEFAULT_RUNS_DIR) / "validation" / "comparisons" / COMPARISON_RUN_ID

for validation_run in VALIDATION_RUNS:
    print(f"Validation run exists: {validation_run.exists()} - {validation_run}")
print(f"Comparison output dir: {COMPARISON_DIR}")

In [ ]:
if RUN_COMPARISON:
    command = [
        "uv",
        "run",
        "face-benchmark",
        "compare-validation-runs",
        "--run-id",
        COMPARISON_RUN_ID,
    ]
    for validation_run in VALIDATION_RUNS:
        command.extend(["--validation-run", str(validation_run)])
    subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Skipping validation comparison. Set RUN_COMPARISON = True to run it.")

In [ ]:
summary_path = COMPARISON_DIR / "summary.md"
precision_recall_overlay_path = COMPARISON_DIR / "precision_recall_overlay.svg"
f_scores_overlay_path = COMPARISON_DIR / "f1_f2_overlay.svg"

print(f"Comparison summary exists: {summary_path.exists()} - {summary_path}")
print(f"Precision-recall overlay exists: {precision_recall_overlay_path.exists()} - {precision_recall_overlay_path}")
print(f"F-score overlay exists: {f_scores_overlay_path.exists()} - {f_scores_overlay_path}")
if summary_path.exists():
    print(summary_path.read_text())